# OhamLab VoiceStudio · Easy Kaggle Hindi Trainer

**Code from your GitHub `main` branch; model and dataset from Hugging Face.** Enable GPU and Internet, set training steps below, then Run All.


## 1 · Training controls

Set **Training steps** (100 for a quick test, 500–1000 for a longer experiment) and run the notebook. You can also change the checkpoint and validation frequency.


In [ ]:
BASE_MODEL = "rahul7star/tts-nano-o-hi"
DATASET_ID = "rahul7star/hindi_dataset_small_example"
GITHUB_REPO = "https://github.com/innokria/ohamlab-voicestudio.git"
GITHUB_BRANCH = "main"  # training code is on main
SPEAKER_ID = None  # automatically selects the speaker with most clips
VOICE_NAME = "OhamHindi"
TRAIN_STEPS = 100  # @param {type:"slider", min:50, max:5000, step:50}
# Number of optimizer steps; start with 100 for a smoke test.
BATCH_FRAMES = 1200
SAVE_EVERY = 50  # @param {type:"integer"}
VALID_EVERY = 50  # @param {type:"integer"}
RESUME = False
RUN_EXPORT = True
UPLOAD_TO_HF = False
HF_OUTPUT_REPO = ""
TEST_SENTENCE = "नमस्ते! यह मेरी नई हिंदी आवाज़ का परीक्षण है।"
WORK = "/kaggle/working/ohamlab"

assert TRAIN_STEPS > 0 and SAVE_EVERY > 0 and VALID_EVERY > 0
print(f"Training: {TRAIN_STEPS} steps | Save every {SAVE_EVERY} | Validate every {VALID_EVERY}")


In [ ]:
# Interactive step control (optional): move the slider before running the training cell.
import ipywidgets as widgets
from IPython.display import display
steps_slider = widgets.IntSlider(value=TRAIN_STEPS, min=50, max=5000, step=50,
    description="Train steps", continuous_update=False, style={"description_width": "initial"},
    layout=widgets.Layout(width="480px"))
def _update_steps(change):
    global TRAIN_STEPS
    TRAIN_STEPS = int(change["new"])
steps_slider.observe(_update_steps, names="value")
display(steps_slider)


## 2 · Install OhamLab directly from GitHub


In [ ]:
import sys, subprocess, pathlib, os
import torch
print("Python:", sys.version.split()[0], "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")
assert torch.cuda.is_available(), "Enable GPU under Kaggle Notebook Settings > Accelerator."

repo_url = f"git+{GITHUB_REPO}@{GITHUB_BRANCH}"
print("Installing from:", repo_url)
# --no-cache-dir avoids stale package installations between notebook revisions.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir", "--upgrade",
                f"ohamlab-voicestudio[train] @ {repo_url}"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "scipy", "librosa",
                "sentencepiece", "speechbrain>=1.0", "torchaudio"], check=True)

# Check the exact module used by later subprocess calls.
check = subprocess.run(
    [sys.executable, "-c",
     "import ohamlab_voicestudio.training.backend.prepare as p; "
     "print('Training backend ready:', p.__file__)"],
    capture_output=True, text=True
)
print(check.stdout or check.stderr)
if check.returncode:
    raise RuntimeError(
        "GitHub branch does not contain the merged training backend. "
        "Push the merged files to branch '" + GITHUB_BRANCH + "' first, "
        "or set GITHUB_BRANCH to the branch where you pushed them.\n"
        + check.stderr
    )


The training package is installed from **your own GitHub repository**. Hugging Face supplies only the base model and Hindi audio dataset. If installation fails, confirm the merged code was pushed to the selected branch.


## 3 · Download model and inspect dataset

In [ ]:
from huggingface_hub import snapshot_download
from datasets import load_dataset, Audio
from collections import Counter
base = snapshot_download(repo_id=BASE_MODEL, local_dir=f"{WORK}/base")
print("Model files:", sorted(p.name for p in pathlib.Path(base).iterdir()))
assert (pathlib.Path(base)/"hima_tts").exists(), "Base model has no hima_tts source; training backend needs compatible inference code."
assert (pathlib.Path(base)/"config.yaml").exists(), "Missing config.yaml in base model."
dset = load_dataset(DATASET_ID, split="train")
print("Rows:", len(dset), "Columns:", dset.column_names)
for c in ("audio", "transcription", "speaker_id"):
    assert c in dset.column_names, f"Missing expected column {c}"
print("Speaker counts:", Counter(str(x) for x in dset["speaker_id"]))

## 4 · Select speaker, preview recordings and prepare audio

In [ ]:
import numpy as np, soundfile as sf
from scipy.signal import resample_poly
from math import gcd
from IPython.display import Audio as AudioPlayer, display
from collections import Counter
counts = Counter(str(x) for x in dset["speaker_id"])
chosen = str(SPEAKER_ID) if SPEAKER_ID is not None else counts.most_common(1)[0][0]
subset = dset.filter(lambda row: str(row["speaker_id"]) == chosen)
print(f"Selected speaker {chosen}: {len(subset)} clips")
assert len(subset) >= 3, "Need at least 3 clips for a useful train/validation pilot. Choose another speaker or larger dataset."
# Force dataset decoding to numpy arrays; this may require audio decoding dependencies on Kaggle.
subset = subset.cast_column("audio", Audio(decode=True))
out = pathlib.Path(WORK)/"clips";out.mkdir(parents=True, exist_ok=True)
rows=[]
for i, item in enumerate(subset):
    text=str(item["transcription"]).strip()
    if not text: continue
    sample=item["audio"]
    audio=np.asarray(sample["array"],dtype=np.float32)
    if audio.ndim==2: audio=audio.mean(axis=0 if audio.shape[0] < audio.shape[1] else 1)
    sr=int(sample["sampling_rate"])
    if sr!=24000:
        g=gcd(sr,24000);audio=resample_poly(audio,24000//g,sr//g).astype(np.float32)
    duration=len(audio)/24000
    if not (0.7<=duration<=29):continue
    name=f"clip_{i:04d}.wav";sf.write(out/name,audio,24000)
    rows.append((name,text,VOICE_NAME))
assert len(rows)>=3, f"Only {len(rows)} valid clips after filtering; need at least 3."
with (out/"metadata.csv").open("w",encoding="utf-8") as f:
    for path,text,voice in rows:
        f.write(f"{path}|{text.replace('|',' ')}|{voice}\n")
print("Prepared",len(rows),"clips. Sample:",rows[0][1]);display(AudioPlayer(filename=str(out/rows[0][0])))

## 5 · Encode dataset for training
This uses the codec and voice embedding model supplied by the compatible base-model runtime. It can take several minutes on first run.

In [ ]:
cmd=[sys.executable,"-m","ohamlab_voicestudio.training.backend.prepare","--base",str(base),"--data",str(out),"--work",WORK]
print("Running:"," ".join(cmd));subprocess.run(cmd,check=True)

## 6 · Fine-tune LoRA adapter (Kaggle GPU)

In [ ]:
TRAIN_STEPS = int(steps_slider.value)  # use current slider value
print(f"Selected training steps: {TRAIN_STEPS}")
cmd=[sys.executable,"-m","ohamlab_voicestudio.training.backend.train","--base",str(base),"--work",WORK,
     "--steps",str(TRAIN_STEPS),"--batch_frames",str(BATCH_FRAMES),"--save_every",str(SAVE_EVERY),"--valid_every",str(VALID_EVERY)]
if RESUME: cmd.append("--resume")
print("Starting training; progress is saved under",WORK+"/run")
subprocess.run(cmd,check=True)

## 7 · Export a compatible voice model and listen

In [ ]:
export_dir=pathlib.Path(WORK)/"exported-model"
if RUN_EXPORT:
    cmd=[sys.executable,"-m","ohamlab_voicestudio.training.backend.export","--base",str(base),"--work",WORK,"--out",str(export_dir),"--samples",TEST_SENTENCE]
    subprocess.run(cmd,check=True)
    wav=export_dir/"samples"/f"{VOICE_NAME}.wav"
    assert wav.exists(), f"No sample was exported: {wav}"
    display(AudioPlayer(filename=str(wav)))
else:
    print("Export skipped")

## 8 · Download results / optional private Hugging Face upload

In [ ]:
import shutil
if export_dir.exists():
    archive=shutil.make_archive("/kaggle/working/ohamlab-hindi-model","zip",root_dir=export_dir)
    print("Download from Kaggle Output:",archive)
if UPLOAD_TO_HF:
    assert HF_OUTPUT_REPO, "Set HF_OUTPUT_REPO in settings"
    from huggingface_hub import HfApi
    token=os.environ.get("HF_TOKEN")
    if not token:
        try:
            from kaggle_secrets import UserSecretsClient
            token=UserSecretsClient().get_secret("HF_TOKEN")
        except Exception as e:
            raise RuntimeError("Add HF_TOKEN as a Kaggle secret; never paste it into the notebook") from e
    api=HfApi(token=token)
    api.create_repo(repo_id=HF_OUTPUT_REPO,repo_type="model",private=True,exist_ok=True)
    api.upload_folder(folder_path=str(export_dir),repo_id=HF_OUTPUT_REPO,repo_type="model")
    print("Uploaded privately:",HF_OUTPUT_REPO)

### Troubleshooting
- **Training backend missing:** Confirm the merged code is pushed to `GITHUB_BRANCH`. The branch name must match exactly.
- **GitHub installation fails:** Enable Internet in Kaggle Settings and verify that the repository/branch is accessible.
- **Too few clips:** Select a speaker with at least three clips or use a larger dataset.
- **Out of memory:** Reduce `BATCH_FRAMES` to 800.
- **Checkpoint resume:** `RESUME=True` requires a saved `WORK/run/state.pt` from the same run.
- **Licensing:** Preserve bundled third-party MIT notices; check model/voice distribution permissions separately.
